# PIEP Modern Synthetic And Noisy Workflows

This notebook focuses on the closed-loop side of the modernization.

The point of the synthetic path is not to replace the archived transcript checks. The point is to stress the internal consistency of the modern implementation:

- choose a known cell
- generate synthetic SAD observations from that cell
- add controlled numerical noise, especially in angular information
- run the same search and post-processing pipeline back on the noisy data

In [ ]:
from dataclasses import replace

import piep
import piep.examples as examples

## Archived-Match Synthetic Patterns

A robust way to generate synthetic regressions is to start from archived source patterns, index them in the known source cell, and then resimulate those same reflection pairs with controlled noise. This keeps the synthetic data scientifically relevant while remaining fully generated by the modern code.

In [ ]:
def archived_like_synthetic_patterns(case, pattern_count=6, positional_sigma_mm=0.01, angle_biases=None, seed=2234):
    if angle_biases is None:
        angle_biases = [0.08, -0.06, 0.04, -0.03, 0.02, -0.01]
    patterns = []
    for index, archived in enumerate(case.patterns[:pattern_count], start=1):
        top_match = piep.index_pattern(archived.observation, case.cell, centering=case.centering).top_match
        maximum_radius_mm = 1.35 * max(
            archived.observation.first_radius,
            archived.observation.second_radius,
            archived.observation.third_radius,
        )
        simulated = piep.simulate_observation_from_zone_pair(
            case.cell,
            piep.ZoneDirection(*top_match.zone_axis),
            top_match.first_hkl,
            top_match.second_hkl,
            title=f'{case.name} synthetic {index}',
            camera_constant=archived.observation.camera_constant,
            maximum_radius_mm=maximum_radius_mm,
            centering=case.centering,
            positional_sigma_mm=positional_sigma_mm,
            reported_radius_sigma_mm=max(
                archived.observation.first_radius_sigma,
                archived.observation.second_radius_sigma,
            ),
            reported_angle_sigma_deg=archived.observation.angle_sigma_deg,
            camera_constant_sigma=archived.observation.camera_constant_sigma,
            high_voltage_volts=archived.observation.high_voltage_volts,
            seed=seed + index - 1,
        )
        observation = replace(
            simulated.observation,
            angle_deg=simulated.observation.angle_deg + angle_biases[index - 1],
        )
        patterns.append(piep.SearchPattern(index, observation))
    return patterns

In [ ]:
lysozyme = examples.lysozyme_case()
synthetic_patterns = archived_like_synthetic_patterns(lysozyme)
synthetic_result = piep.search_and_conventionalize(
    synthetic_patterns,
    volume_min=lysozyme.exact_volume,
    volume_max=lysozyme.exact_volume,
    centering=lysozyme.centering,
    increment_value=0.01,
    preferred_centering=lysozyme.preferred_centering,
    minimum_system='triclinic',
    top_n=20,
)

for index, candidate in enumerate(synthetic_result.candidates, start=1):
    preferred = candidate.preferred_candidate
    if preferred is None:
        continue
    print(index, preferred.system, preferred.centering, preferred.cell)

## Why This Matters

This synthetic loop is especially useful for post-processing work.

Small angle perturbations can push noisy candidate cells toward nearby reduced or pseudo-higher-symmetry branches. A good modern implementation should still recover a scientifically meaningful conventional description from those noisy candidates.

That is why the current regression strategy uses both archived transcript checks and noisy synthetic checks:

- archived data verifies fidelity to the published workflows
- synthetic data verifies internal consistency and numerical robustness